### html + htmlheadertextsplitter+ remove html junk + better chunking

### different embedding models + retrieval strategies

In [1]:
import sys
import os
import chromadb
path = "C:\\Users\\luyer\\my_rag\\data"

In [2]:
client = chromadb.PersistentClient(path=path)

In [3]:
from chromadb.utils.embedding_functions import (
    SentenceTransformerEmbeddingFunction
)

mpnet_ef = SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2",
    device="cpu",
    normalize_embeddings=True,
)

mpnet_collection = client.create_collection(
    name="my_movies_mpnet_cosine",
    embedding_function=mpnet_ef,
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\luyer\my_rag\.my_rag\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\luyer\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
files = [
    file_name
    for file_name in os.listdir(path)
    if os.path.isfile(os.path.join(path, file_name)) and file_name.endswith('.html')
]
print(files)

['2001_A_Space_Odyssey.html', 'Casablanca.html', 'Citizen_Kane.html', 'Parasite_2019.html', 'Pulp_Fiction.html', 'Seven_Samurai.html', 'Spirited_Away.html', 'The_Dark_Knight.html', 'The_Godfather.html', 'The_Matrix.html']


In [5]:
import re
from bs4 import BeautifulSoup
from pathlib import Path
from langchain_text_splitters import (
    HTMLHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

def clean_chunk_text(text: str) -> str:
    """Clean text after HTML parsing."""
    text = text.replace("\x00", "")

    # remove Wikipedia citation markers like [1], [22], [123]
    text = re.sub(r"\[\d+\]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

header_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=[
        ("h1", "header1"),
        ("h2", "header2"),
        ("h3", "header3"),
    ]
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "? ",
        "! ",
        " ",
        "",
    ]
)

BAD_SECTIONS = {
    "References",
    "External links",
    "Further reading",
}

all_chunks = []

for file_name in files:
    file_path = Path(path) / file_name

    raw_html = file_path.read_text(encoding="utf-8", errors="ignore")

    # ----------------------------
    # 1. Parse HTML
    # ----------------------------

    soup = BeautifulSoup(raw_html, "html.parser")

    # remove useless HTML elements
    for tag in soup([
        "script",
        "style",
        "nav",
        "footer",
        "noscript",
    ]):
        tag.decompose()

    # ----------------------------
    # 2. Keep article body only
    # ----------------------------

    content = soup.select_one("#mw-content-text")

    if content is None:
        content = soup.body

    if content is None:
        continue

    clean_html = str(content)

    # ----------------------------
    # 3. Split by HTML headings
    # ----------------------------

    section_chunks = header_splitter.split_text(clean_html)
    
    # ----------------------------
    # 4. Exclude bad sections
    # ----------------------------

    section_chunks = [
        doc
        for doc in section_chunks
        if doc.metadata.get("header2") not in BAD_SECTIONS
    ]

    chunks = text_splitter.split_documents(section_chunks)
    # ----------------------------
    # 6. Clean + enrich chunks
    # ----------------------------

    for chunk_index, chunk in enumerate(chunks):

        text = clean_chunk_text(chunk.page_content)

        if not text:
            continue

        h1 = chunk.metadata.get("header1", "")
        h2 = chunk.metadata.get("header2", "")
        h3 = chunk.metadata.get("header3", "")

        # Add structural context into embedded text
        prefix_parts = []

        if h1:
            prefix_parts.append(f"Article: {h1}")

        if h2:
            prefix_parts.append(f"Section: {h2}")

        if h3:
            prefix_parts.append(f"Subsection: {h3}")

        if prefix_parts:
            text = "\n".join(prefix_parts) + "\n\n" + text

        all_chunks.append(
            {
                "id": f"html-{file_name}-{chunk_index}",
                "document": text,
                "metadata": {
                    **chunk.metadata,
                    "source": file_name,
                },
            }
        )




In [6]:
mpnet_collection.add(
    ids=[item["id"] for item in all_chunks],
    documents=[item["document"] for item in all_chunks],
    metadatas=[item["metadata"] for item in all_chunks],
)

print(f"Added {len(all_chunks)} HTML chunks")

Added 2327 HTML chunks


In [7]:
import pprint

In [8]:
pprint.pprint(chunk.page_content)

('Against Technology: From the Luddites to Neo-Luddism  \n'
 'ISBN  \n'
 '978-0-415-97868-2  \n'
 'Pegg, Simon (2010). . Century. .  \n'
 'Nerd Do Well  \n'
 'ISBN  \n'
 '978-1-84605-811-0  \n'
 'Toropov, Brandon; Hansen, Chad (2002). . Penguin. .  \n'
 "The Complete Idiot's Guide to Taoism  \n"
 'ISBN  \n'
 '978-0-02-864262-8  \n'
 'Wachowski, Larry; Wachowski, Andy (2000). . Titan. .  \n'
 'The Art of The Matrix  \n'
 'ISBN  \n'
 '978-1-84023-173-1  \n'
 'Wood, Aylish (2007). . Routledge. .  \n'
 'Digital Encounters  \n'
 'ISBN  \n'
 '978-0-415-41066-3  \n'
 '{{DEFAULTSORT:Matrix, The}} would sort inappropriately after The Matrix '
 'Reloaded and The Matrix Revolutions')


In [9]:
len(chunks)

234

In [10]:
chunks[0].page_content

'1999 film by the Wachowskis  \nThis article is about the 1999 film. For the franchise it initiated, see . For other uses, see .  \n(franchise)  \nThe Matrix  \nMatrix  \nThe Matrix  \nTheatrical release poster  \nDirected by  \nThe Wachowskis  \na  \n[  \n]  \nWritten by  \nThe Wachowskis  \nProduced by  \nJoel Silver  \nStarring  \nKeanu Reeves  \nLaurence Fishburne  \nCarrie-Anne Moss  \nHugo Weaving  \nJoe Pantoliano  \nCinematography  \nBill Pope  \nEdited by  \nZach Staenberg  \nMusic by  \nDon Davis  \nProduction companies  \nVillage Roadshow Pictures  \nGroucho II Film Partnership  \nSilver Pictures'

In [25]:
type(chunks)

list

In [12]:
mpnet_collection.query(
    query_texts=["What year was 2001: A Space Odyssey released, and what genre is it?"],
    n_results=15
)

{'ids': [['html-2001_A_Space_Odyssey.html-296',
   'html-2001_A_Space_Odyssey.html-6',
   'html-2001_A_Space_Odyssey.html-260',
   'html-2001_A_Space_Odyssey.html-293',
   'html-2001_A_Space_Odyssey.html-2',
   'html-2001_A_Space_Odyssey.html-149',
   'html-2001_A_Space_Odyssey.html-57',
   'html-2001_A_Space_Odyssey.html-201',
   'html-2001_A_Space_Odyssey.html-156',
   'html-2001_A_Space_Odyssey.html-0',
   'html-2001_A_Space_Odyssey.html-177',
   'html-2001_A_Space_Odyssey.html-261',
   'html-2001_A_Space_Odyssey.html-267',
   'html-2001_A_Space_Odyssey.html-295',
   'html-2001_A_Space_Odyssey.html-255']],
 'embeddings': None,
 'documents': [["Section: Sources\n\nThe Making of 2001: A Space Odyssey Jay Cocks Random House ISBN 978-0-307-75760-9 Archived . Retrieved 2020 21 September Walker, Alexander (1971). . New York: Harcourt Brace Jovanovich. . Stanley Kubrick Directs ISBN 0-393-32119-3 Note: This is a revised edition of . Walker, Alexander (2000). . New York: W. W. Norton and Co

In [13]:
cosine_collection.query(
    query_texts=["Who is the director of 2001: A Space Odyssey?"],
    n_results=15
)

{'ids': [['html-2001_A_Space_Odyssey.html-6',
   'html-2001_A_Space_Odyssey.html-293',
   'html-2001_A_Space_Odyssey.html-149',
   'html-2001_A_Space_Odyssey.html-260',
   'html-2001_A_Space_Odyssey.html-205',
   'html-2001_A_Space_Odyssey.html-296',
   'html-2001_A_Space_Odyssey.html-57',
   'html-2001_A_Space_Odyssey.html-94',
   'html-2001_A_Space_Odyssey.html-7',
   'html-2001_A_Space_Odyssey.html-201',
   'html-2001_A_Space_Odyssey.html-0',
   'html-2001_A_Space_Odyssey.html-2',
   'html-2001_A_Space_Odyssey.html-264',
   'html-2001_A_Space_Odyssey.html-156',
   'html-2001_A_Space_Odyssey.html-200']],
 'embeddings': None,
 'documents': [["2001: A Space Odyssey Uptown Theater Metro-Goldwyn-Mayer 2001: A Space Odyssey human evolution technology artificial intelligence extraterrestrial life Academy Awards visual effects 5 [ ] The film is widely regarded as one of the . In 1991, it was selected by the United States for preservation in the . In 2022, placed in the top ten of s decennia

In [14]:
collection_persistent.query(
    query_texts=["What year was 2001: A Space Odyssey released, and what genre is it?"],
)

{'ids': [['html-2001_A_Space_Odyssey.html-164',
   'html-2001_A_Space_Odyssey.html-433',
   'html-2001_A_Space_Odyssey.html-99',
   'html-2001_A_Space_Odyssey.html-375',
   'html-2001_A_Space_Odyssey.html-223',
   'html-2001_A_Space_Odyssey.html-434',
   'html-2001_A_Space_Odyssey.html-220',
   'html-2001_A_Space_Odyssey.html-10',
   'html-2001_A_Space_Odyssey.html-171',
   'html-2001_A_Space_Odyssey.html-395']],
 'embeddings': None,
 'documents': [['Main article: 2001: A Space Odyssey (soundtrack)',
   "v t e Space Odyssey Films (1968) 2001: A Space Odyssey (1984) 2010: The Year We Make Contact Novels (1968) 2001: A Space Odyssey (1982) 2010: Odyssey Two (1987) 2061: Odyssey Three (1997) 3001: The Final Odyssey Non-fiction The Lost Worlds of 2001 Comics 2001: A Space Odyssey Characters HAL 9000 Elements Monoliths Discovery Related Interpretations of 2001: A Space Odyssey Technologies in 2001: A Space Odyssey in popular culture 2001: A Space Odyssey soundtrack 2001: A Space Odyssey Ale